In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42


In [2]:
df_all = pd.read_csv("train_with_clusters.csv")
print(f"Loaded train_with_clusters.csv (Shape: {df_all.shape})")

Loaded train_with_clusters.csv (Shape: (5807, 98))


In [3]:
preprocess_obj = joblib.load("preprocess_for_clustering.joblib")
scaler = preprocess_obj["scaler"]
    # Based on the typical saving format, the key is assumed to be 'feature_names'
feature_names = preprocess_obj["feature_names"]
print(f"Loaded preprocessing objects (Scaler and {len(feature_names)} features).")

Loaded preprocessing objects (Scaler and 40 features).


In [4]:
top_40 = joblib.load("top_features_for_clustering.joblib")
feature_names = top_40
print(f"Loaded preprocessing objects (Scaler and {len(feature_names)} features).")


Loaded preprocessing objects (Scaler and 40 features).


In [5]:
X_raw = df_all[feature_names]
y_cluster = df_all["cluster"]

In [6]:
X_cluster_train = scaler.transform(X_raw)

In [7]:
print("-" * 60)
print(f"Training Cluster Classifier on {X_cluster_train.shape[0]} rows.")

------------------------------------------------------------
Training Cluster Classifier on 5807 rows.


In [8]:
clf_cluster = RandomForestClassifier(
    n_estimators=300,        # Increased estimators for better accuracy
    max_depth=20,            # Deeper trees to capture complex cluster shapes
    criterion='entropy',     # Often better than 'gini' for multi-class problems
    class_weight='balanced', # Crucial for handling small/imbalanced clusters (e.g., C2, C3, C6)
    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [9]:
clf_cluster.fit(X_cluster_train, y_cluster)
y_pred_cluster = clf_cluster.predict(X_cluster_train)
acc = accuracy_score(y_cluster, y_pred_cluster)

In [10]:
print("SECTION 3.3.1: CLUSTER-ID PREDICTION MODEL RESULTS")
print("-" * 60)
print(f"Model: RandomForestClassifier (Trained on 40 scaled features)")
print(f"Overall Train Accuracy: {acc:.4f}")
print("\nClassification Report (Per-Cluster Performance, critical for generalization):\n")
print(classification_report(y_cluster, y_pred_cluster, zero_division=0))

SECTION 3.3.1: CLUSTER-ID PREDICTION MODEL RESULTS
------------------------------------------------------------
Model: RandomForestClassifier (Trained on 40 scaled features)
Overall Train Accuracy: 1.0000

Classification Report (Per-Cluster Performance, critical for generalization):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       728
           1       1.00      1.00      1.00       701
           2       1.00      1.00      1.00      2059
           3       1.00      1.00      1.00         1
           4       1.00      1.00      1.00      2313
           5       1.00      1.00      1.00         2
           6       1.00      1.00      1.00         3

    accuracy                           1.00      5807
   macro avg       1.00      1.00      1.00      5807
weighted avg       1.00      1.00      1.00      5807



In [11]:
importances = clf_cluster.feature_importances_
indices = np.argsort(importances)[::-1]

print("\nTop 5 Features Separating the Clusters:")
for i in range(5):
    print(f"{i+1}. {feature_names[indices[i]]} (Importance: {importances[indices[i]]:.4f})")


Top 5 Features Separating the Clusters:
1.  Per Share Net profit before tax (Yuan ¥) (Importance: 0.0706)
2.  Net profit before tax/Paid-in capital (Importance: 0.0641)
3.  Net Income to Stockholder's Equity (Importance: 0.0595)
4.  After-tax net Interest Rate (Importance: 0.0571)
5.  Equity to Liability (Importance: 0.0557)


In [12]:
cluster_id_package = {
    "model": clf_cluster,
    "features": feature_names,
    "scaler": scaler 
}

joblib.dump(cluster_id_package, "cluster_4_model.joblib")
print("Saved cluster_id_model.joblib for use by all team members in Section 4.")

Saved cluster_id_model.joblib for use by all team members in Section 4.
